In [3]:
#####################################
# Chapter 3: 어텐션 메커니즘 (모델의 핵심)
#####################################
class MultiHeadAttention(nn.Module):
    """
    멀티 헤드 셀프 어텐션 (Multi-Head Self-Attention) 모듈입니다.
    입력 데이터 간의 관계성을 여러 관점(Head)에서 병렬로 학습합니다.
    """
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "출력 차원(d_out)은 헤드 수(num_heads)로 나누어 떨어져야 합니다."

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads  # 각 헤드가 담당할 차원 크기

        # Query, Key, Value를 만들기 위한 선형 투영 레이어들
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

        # 멀티 헤드 결과를 하나로 합친 후 통과시키는 출력 레이어
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)

        # Causal Mask (인과적 마스킹) 생성: 미래의 토큰을 보지 못하게 함
        # 상삼각 행렬(대각선 위쪽)을 1로 채워서 나중에 마스킹에 사용
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape # b: 배치 크기, num_tokens: 시퀀스 길이

        # 1. Q, K, V 계산
        keys = self.W_key(x)     # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # 2. 헤드 나누기 (Multi-head splitting)
        # 차원을 변형하여 여러 헤드가 병렬로 처리할 수 있게 함
        # (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # 3. 차원 순서 변경 (Transpose)
        # (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        # 이렇게 하면 (num_tokens, head_dim) 행렬이 헤드 개수만큼 독립적으로 존재하게 됨
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # 4. Scaled Dot-Product Attention 계산
        # Query와 Key의 내적 (유사도 계산)
        # 결과 Shape: (b, num_heads, num_tokens, num_tokens)
        attn_scores = ### queries @ keys.transpose(2, 3) ## Write Answer

        # 5. 마스킹 (Masking)
        # 현재 시점보다 미래의 토큰 정보를 참조하지 못하게 -inf로 가림
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        # 6. 소프트맥스 및 드롭아웃
        # 점수를 확률로 변환 (합이 1이 되도록)
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # 7. Value와의 가중치 합 (Context Vector 계산)
        # Shape: (b, num_heads, num_tokens, head_dim)
        context_vec = ### (attn_weights @ values).transpose(1, 2) ## Write Answer

        # 8. 헤드 결합 (Concatenation)
        # 나눠졌던 헤드들을 다시 원래의 d_out 차원으로 합침
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)

        # 9. 최종 선형 투영
        context_vec = self.out_proj(context_vec)

        return context_vec

SyntaxError: invalid syntax (924216514.py, line 55)

In [2]:
##중요 : 아래는 GPT 모델 전체 구조를 정의하는 코드입니다

#####################################
# Chapter 4: GPT 아키텍처 조립
#####################################
class TransformerBlock(nn.Module):
    """
    표준 트랜스포머 블록 (Decoder Block)
    구조: LayerNorm -> Attention -> Add(Residual) -> LayerNorm -> FeedForward -> Add(Residual)
    """
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"])
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        # region [어텐션 블록 (Residual Connection 적용)]
        shortcut = x
        x = self.norm1(x) # Pre-LayerNorm 방식
        x = self.att(x)
        x = self.drop_shortcut(x)
        x = x + shortcut  # 원본 입력을 더해줌 (기울기 소실 방지)
        # endregion

        # region [피드 포워드 블록 (Residual Connection 적용)]
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut  # 원본 입력 다시 더함
        # endregion

        return x
class GPTModel(nn.Module):
    """
    전체 GPT 모델 구조 정의
    Embedding -> Transformer Blocks -> Final Norm -> Output Head
    """
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = ### nn.Embedding(cfg["vocab_size"], cfg["emb_dim"]) ###  !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!
        self.pos_emb = ### nn.Embedding(cfg["context_length"], cfg["emb_dim"]) ## Write Answer
        self.drop_emb = ### nn.Dropout(cfg["drop_rate"]) ## Write Answer

        # 트랜스포머 블록 쌓기 (n_layers 만큼)
        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])

        # 최종 정규화 및 출력 헤드
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape

        # region [임베딩 생성]
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        # endregion

        # region [토큰 임베딩과 위치 임베딩 합산]
        x = tok_embeds + pos_embeds
        # endregion

        x = self.drop_emb(x)

        # region [트랜스포머 블록 통과]
        x = self.trf_blocks(x)
        # endregion

        # region [최종 출력 계산]
        x = self.final_norm(x)
        logits = self.out_head(x) # 각 단어에 대한 예측 점수 (Logits)
        # endregion

        return logits



def generate_text_simple(model, idx, max_new_tokens, context_size):
    """
    간단한 텍스트 생성 루프
    현재 문맥을 넣어 다음 토큰을 예측하고, 이를 다시 문맥에 추가하여 반복함
    """
    # idx: 현재 문맥의 토큰 인덱스들 (Batch, Time)
    for _ in range(max_new_tokens):

        # 모델이 지원하는 최대 길이(context_size)를 넘지 않도록 자름
        idx_cond = idx[:, -context_size:]

        # 모델 예측 (기울기 계산 불필요) -> 기울기는 역전파에서 사용함, 역전파는 학습시에만 필요해서 추론에서는 필요 없기 대문에 no_grad로 설정
        with torch.no_grad():
            logits = model(idx_cond)

        # 마지막 타임스텝의 예측값만 가져옴 (다음 단어 예측이므로)
        # (batch, n_token, vocab_size) -> (batch, vocab_size)
        logits = ### logits[:, -1, :] ## Write Answer

        # 가장 확률(로짓값)이 높은 토큰 선택 (Greedy Decoding)
        idx_next = ### torch.argmax(logits, dim=-1, keepdim=True) ## Write Answer

        # 예측된 토큰을 현재 시퀀스 뒤에 이어 붙임
        idx = torch.cat((idx, idx_next), dim=1)

    return idx

SyntaxError: invalid syntax (2749542341.py, line 50)

In [1]:
# [DETR] 입력 전처리 + 박스 좌표 후처리 함수
# - DETR은 COCO로 학습된 ResNet backbone을 사용하므로, ImageNet mean/std 정규화를 맞춰야 합니다.
# - 모델 출력 박스는 (cx, cy, w, h) 형식이며 0~1로 정규화되어 있으므로, (xmin, ymin, xmax, ymax)로 변환 후 원본 이미지 크기로 rescale 합니다.

# standard PyTorch mean-std input image normalization
transform = T.Compose([  # 전처리 파이프라인 정의(논문/코드: data augmentation과 정규화)
    ### T.Resize(800) , ## Write Answer
    ### T.ToTensor(), ## Write Answer
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# for output bounding box post-processing
def box_cxcywh_to_xyxy(x):  # (cx,cy,w,h) → (xmin,ymin,xmax,ymax) 변환 # jm - 박스를 좌표로
    x_c, y_c, w, h = x.unbind(1)  # 각 성분을 분리: x.shape=(N,4) 가정
    b = ### [(x_c - 0.5 * w), (y_c - 0.5 * h), (x_c + 0.5 * w), (y_c + 0.5 * h)] ## Write Answer
    return torch.stack(b, dim=1)  # (N,4)로 다시 스택

def rescale_bboxes(out_bbox, size):  # 정규화 박스를 픽셀 좌표로 변환
    img_w, img_h = size  # PIL size = (W,H)
    b = box_cxcywh_to_xyxy(out_bbox)  # 우선 xyxy로 변환
    b = b * torch.tensor([img_w, img_h, img_w, img_h], dtype=torch.float32)  # (W,H,W,H) 스케일로 곱해 픽셀 단위로 확장
    return b

SyntaxError: invalid syntax (1742505088.py, line 15)